# statsmodels

A refresher on **statsmodels** — Python's library for **classical statistical modeling and
inference**. Where scikit-learn asks "what's the best prediction?", statsmodels asks "what is the
*relationship*, and how sure are we?" — it hands you coefficient estimates with standard errors,
p-values, confidence intervals, R², residual diagnostics, and a printed regression table that
looks like the output of R or Stata. Its home turf: OLS/GLM regression, time series
(ARIMA/SARIMAX, state space), and a large bag of statistical tests.

For the broader statistics-method overview see [[statistical-modeling]]; for the array/DataFrame
substrate it sits on see [[numpy-pandas-scipy]]; for mixed-effects / hierarchical models see
[[mixed-effects-models]] and [[lme4]]; for the Bayesian take on the same regression problems see
[[pymc]].

**Domain:** Data Analysis & Research  ·  **recommended addition**  ·  **runnable:** yes

## 1. What & Why

**What it is.** statsmodels is a Python package for **estimating statistical models and doing
inference on them**. You fit a model — `OLS`, `GLM`, `Logit`, `ARIMA`, `MixedLM`, … — and get
back a *results object* whose `.summary()` is a full inferential report: point estimates, standard
errors, t/z statistics, p-values, confidence intervals, goodness-of-fit (R², AIC/BIC,
log-likelihood), and diagnostic tests. It also ships a deep library of stand-alone statistical
tests and tools (`statsmodels.stats`, `statsmodels.tsa`).

**The problem it solves.** scikit-learn is built to *predict* — `.fit()/.predict()`, cross-validated
accuracy, no p-values in sight. But a huge amount of data work is *explanation and inference*: Is
this coefficient significantly different from zero? How big is the effect, and what's its 95%
interval? Does treatment matter after controlling for age? Is this time series trending or just
noisy? statsmodels gives you the estimator **plus the uncertainty around it**, in the vocabulary
of classical (frequentist) statistics, with the formula interface (`y ~ x + C(group)`) familiar
from R.

**When to reach for it.** Regression where you care about *which variables matter and by how much*
(econometrics, A/B test analysis, scientific modeling); generalized linear models (logistic,
Poisson); classical time-series modeling and forecasting (ARIMA/SARIMAX); and hypothesis tests
(t-tests, ANOVA, normality/heteroskedasticity/stationarity tests). **When not to:** you only want
the best black-box predictor and don't need inference (use [[numpy-pandas-scipy]] + scikit-learn);
you need deep nets or GPUs (PyTorch); or you want full Bayesian posteriors and priors (use
[[pymc]]).

## 2. Mental Model

**statsmodels is "R's `lm()`/`glm()`, in Python." You specify a model — often with a formula
string — fit it, and the payoff is the `summary()` table: not just predictions, but *every number a
statistician wants to judge the fit*. The result object is the product, and inference (SEs,
p-values, CIs) is first-class, not an afterthought.**

The workflow is always the same three steps:

1. **Specify** the model. Two equivalent front doors:
   - **Formula API** (`statsmodels.formula.api as smf`): `smf.ols("y ~ x + C(g)", data=df)` — R-style
     Patsy/formulaic strings over a DataFrame; categoricals, interactions (`x*z`), and transforms
     (`np.log(x)`) are handled for you, and an intercept is added automatically.
   - **Arrays API** (`statsmodels.api as sm`): `sm.OLS(y, X)` — you pass NumPy/pandas arrays and
     **must add the intercept yourself** with `sm.add_constant(X)`.
2. **Fit** it: `.fit()` returns a *results* object (estimation is immediate for linear models,
   iterative for GLM/ARIMA).
3. **Interrogate** the results: `.summary()` for the human report; `.params`, `.bse`, `.pvalues`,
   `.conf_int()`, `.predict()` for programmatic access; residual/diagnostic tests for model checking.

If scikit-learn is "fit → predict → score on held-out data," statsmodels is "specify → fit →
*explain*": the deliverable is understanding the data-generating process, with calibrated
uncertainty attached to every estimate.

## 3. Key Concepts

- **`endog` and `exog`.** statsmodels' names for the dependent variable (*endog*enous, the `y`) and
  the regressors (*exog*enous, the design matrix `X`). The formula API builds both for you from
  `y ~ x` terms; the arrays API makes you pass them.
- **Two APIs.** `statsmodels.formula.api` (lowercase: `ols`, `glm`, `logit`) takes a **formula +
  DataFrame** and adds the intercept automatically. `statsmodels.api` (uppercase: `OLS`, `GLM`,
  `Logit`) takes **arrays** and requires `sm.add_constant(X)` for an intercept. Pick one; mixing up
  the intercept rule is the classic beginner bug.
- **Model vs Results.** `smf.ols(...)` (or `sm.OLS(...)`) builds an unfitted **model**; `.fit()`
  returns a **results** object. Everything you want — params, inference, predictions, diagnostics —
  lives on the *results*.
- **The `summary()` table.** The signature output. Top block: R²/adj-R², F-stat, AIC/BIC,
  log-likelihood, n obs. Coefficient block: `coef`, `std err`, `t`/`z`, `P>|t|`, and the
  `[0.025  0.975]` confidence interval per term. Bottom block: residual diagnostics (Durbin-Watson,
  Jarque-Bera, condition number).
- **GLM and families.** `GLM(y, X, family=...)` generalizes OLS via a link function and error
  distribution: `Binomial` (logistic), `Poisson` (counts), `Gamma`, etc. Coefficients are on the
  *link* scale — exponentiate Binomial coefficients to read odds ratios.
- **Robust / clustered standard errors.** `.fit(cov_type="HC3")` (heteroskedasticity-robust) or
  `cov_type="cluster"` change *how SEs are computed* without changing the point estimates — crucial
  when the constant-variance assumption fails.
- **`tsa` — time series.** `statsmodels.tsa` holds `ARIMA`/`SARIMAX`, state-space models,
  `seasonal_decompose`, ACF/PACF, and tests like `adfuller` (stationarity). Forecasting is
  `.get_forecast(steps)` / `.forecast(steps)`.
- **`statsmodels.stats`.** A standalone test library: t-tests, ANOVA (`anova_lm`), proportions,
  power analysis, multiple-comparison correction, and diagnostics (`het_breuschpagan`,
  `durbin_watson`).

## 4. Setup

Pure-Python wheel (built on NumPy/SciPy/pandas), CPU-only, installs in seconds. The formula API
also pulls in `patsy`/`formulaic` for the `y ~ x` strings. No server, no GPU, no data downloads
needed for anything below — every example uses small simulated data.

In [ ]:
# %pip install statsmodels pandas numpy
import numpy as np
import pandas as pd
import statsmodels.api as sm            # arrays API + families/datasets
import statsmodels.formula.api as smf   # R-style formula API

print("statsmodels", sm.__version__)
print("pandas     ", pd.__version__)

# Smallest possible fit: recover a slope from a straight line + noise.
rng = np.random.default_rng(0)
x = rng.normal(size=50)
y = 3.0 + 2.0 * x + rng.normal(scale=0.5, size=50)
quick = sm.OLS(y, sm.add_constant(x)).fit()   # add_constant -> intercept term
print("\nintercept, slope =", quick.params.round(3))

## 5. Worked Examples

Everything below runs in a fresh kernel on simulated data — no downloads, no API keys, CPU-only.
We cover the three pillars: **OLS with full inference**, a **GLM (logistic) regression**, and a
**time-series ARIMA fit** — plus a gated example showing how to pull a real dataset.

### Example 1 — OLS via the formula API, and reading off inference

We simulate data from a *known* model, then recover the coefficients and confirm the inference
machinery flags the true effects as significant.

In [ ]:
n = 200
df = pd.DataFrame({
    "x": rng.normal(size=n),
    "group": rng.choice(["A", "B"], size=n),
})
# True model:  y = 1.5 + 2.0*x - 1.0*(group == "B") + noise
df["y"] = 1.5 + 2.0 * df["x"] - 1.0 * (df["group"] == "B") + rng.normal(scale=0.5, size=n)

# Formula API: C(group) makes a categorical (B vs the A baseline); intercept is automatic.
model = smf.ols("y ~ x + C(group)", data=df).fit()
print(model.summary())

The `coef` column recovers the true `1.5 / 2.0 / -1.0`, every `P>|t|` is ~0 (all real effects),
and `R-squared` is high because the noise is small. Now pull the same numbers out
**programmatically** — that's how you use a fit in code rather than by eye.

In [ ]:
print("coefficients:")
print(model.params.round(3), "\n")

print("p-values:")
print(model.pvalues.round(4), "\n")

print("95% confidence intervals:")
print(model.conf_int().round(3), "\n")

print("R^2 =", round(model.rsquared, 3))

# Predict for brand-new rows (formula handles the categorical encoding for you).
newdata = pd.DataFrame({"x": [0.0, 1.0], "group": ["A", "B"]})
print("\npredictions:", model.predict(newdata).round(3).tolist())

### Example 2 — Logistic regression as a GLM

Binary outcome → `GLM` with the `Binomial` family (a.k.a. logistic regression). Coefficients live
on the **log-odds** scale, so exponentiate to get an odds ratio.

In [ ]:
# Simulate a binary outcome whose log-odds depend linearly on x.
log_odds = -0.5 + 1.8 * df["x"]
df["click"] = rng.binomial(1, 1 / (1 + np.exp(-log_odds)))

logit = smf.glm("click ~ x", data=df, family=sm.families.Binomial()).fit()
print(logit.summary().tables[1])   # just the coefficient table

# Coefficients are log-odds; exp() turns the x coefficient into an odds ratio.
print("\nodds ratio for a 1-unit increase in x:", round(np.exp(logit.params["x"]), 3))
print("(equivalently, smf.logit('click ~ x', data=df).fit() gives the same fit)")

### Example 3 — Time series: fit an ARIMA model and forecast

`statsmodels.tsa` is the classical time-series toolkit. We generate an AR(1) process, fit
`ARIMA(1,0,0)`, and forecast a few steps ahead with confidence intervals.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# AR(1): each value is 0.7 * previous + noise.
vals = [0.0]
for _ in range(200):
    vals.append(0.7 * vals[-1] + rng.normal(scale=0.3))
ts = pd.Series(vals)

arima = ARIMA(ts, order=(1, 0, 0)).fit()   # (p, d, q) = (1 AR lag, no differencing, 0 MA)
print(arima.summary().tables[1])

fc = arima.get_forecast(steps=3)
print("\nforecast (next 3):", fc.predicted_mean.round(3).tolist())
print("95% CI:")
print(fc.conf_int().round(3))

The fitted `ar.L1` coefficient lands near the true 0.7. `get_forecast` returns both the point
forecast and the interval — the band widens as you predict further out, which is exactly the
honest behavior you want from a forecaster.

### Example 4 — Pulling a real dataset (gated)

statsmodels can fetch real datasets from the R datasets repo via `get_rdataset()`, but that needs
network access — so it's gated behind an env check. The notebook still runs end-to-end without it
while showing the call shape.

In [ ]:
import os

if os.getenv("STATSMODELS_FETCH"):
    # mtcars: classic regression demo — fuel economy vs weight and horsepower.
    mtcars = sm.datasets.get_rdataset("mtcars").data
    fit = smf.ols("mpg ~ wt + hp", data=mtcars).fit()
    print(fit.params.round(3))
else:
    print("Set STATSMODELS_FETCH=1 to pull mtcars via get_rdataset() (needs network).")
    print("Shape:")
    print("  data = sm.datasets.get_rdataset('mtcars').data")
    print("  smf.ols('mpg ~ wt + hp', data=data).fit().summary()")

## 6. Gotchas & Pitfalls

- **The intercept rule differs by API.** The **formula** API (`smf.ols`, lowercase) adds an
  intercept automatically; the **arrays** API (`sm.OLS`, uppercase) does **not** — you must wrap
  `X` in `sm.add_constant(X)` or you silently fit a no-intercept (through-the-origin) model. This
  is the single most common statsmodels bug.
- **`exog` order is `(endog, exog)` for arrays.** `sm.OLS(y, X)` — outcome first, regressors
  second. Easy to flip if you're used to sklearn's `fit(X, y)`.
- **statsmodels ≠ scikit-learn.** No `.fit_transform`, no built-in cross-validation, no
  `predict_proba`. The point estimates aren't regularized by default (it's classical MLE/OLS); for
  L1/L2 use `.fit_regularized` or reach for sklearn when prediction is the goal.
- **GLM coefficients are on the link scale.** A `Binomial` model's coefficients are log-odds, a
  `Poisson` model's are log-rates. Exponentiate before interpreting as odds/rate ratios — don't read
  them as probabilities.
- **Default SEs assume homoskedasticity + independence.** If errors have non-constant variance or
  clustering, your p-values are wrong. Refit with `cov_type="HC3"` (robust) or
  `cov_type="cluster"`, which fixes inference without changing the coefficients.
- **A high condition number means collinearity.** `summary()` warns at the bottom when regressors
  are nearly collinear (or just unscaled). It inflates standard errors; center/scale predictors or
  drop redundant terms.
- **Missing data is dropped silently-ish.** The default `missing="none"` errors on NaNs; the formula
  API uses `missing="drop"`. Know which rows were dropped — `model.nobs` tells you how many actually
  fit.
- **ARIMA wants a clean, ordered series.** Pass a `Series` (ideally with a `DatetimeIndex` and a set
  frequency) sorted in time; choose `d` to make it stationary (check `adfuller`) before trusting
  `(p, d, q)`. A wrong `d` gives nonsense forecasts.
- **`predict` vs `get_prediction`.** `.predict()` returns point estimates only; `.get_prediction()`
  (regression) / `.get_forecast()` (tsa) return an object with `.conf_int()` for the uncertainty
  band.

## 7. When to Use vs Alternatives

| Option | Best at | Reach for it instead of statsmodels when… |
| --- | --- | --- |
| **statsmodels** | Inference: regression/GLM/time-series with p-values, CIs, diagnostics; classical stats tests | — |
| **scikit-learn** | Predictive ML: pipelines, cross-validation, regularization, many estimators | You want the best black-box predictor and don't need p-values or coefficient inference |
| **[[pymc]]** | Full Bayesian inference: priors, posteriors, hierarchical models, uncertainty by sampling | You want a posterior distribution and to encode prior knowledge, not a frequentist point estimate |
| **[[lme4]] / [[mixed-effects-models]]** | Mixed / hierarchical models with complex random-effects structures (R's gold standard) | You need rich random-effects specs; statsmodels `MixedLM` is more limited than lme4 |
| **R (`lm`/`glm`/`forecast`)** | The reference implementation for classical stats; vast modeling ecosystem | You live in R, or need a model statsmodels hasn't implemented — see [[r-language]] |
| **NumPy/SciPy** | Hand-rolled estimators, raw linear algebra, individual `scipy.stats` tests | You need one specific test/distribution and not a full modeling framework — see [[numpy-pandas-scipy]] |

**Rules of thumb.** Need a coefficient *with* a p-value and confidence interval → statsmodels. Need
the most accurate prediction on held-out data → scikit-learn. Need priors and a posterior →
[[pymc]]. Complex grouped/hierarchical random effects → [[lme4]] (or statsmodels `MixedLM` for
simpler cases). statsmodels and scikit-learn are complements: prototype and explain in statsmodels,
productionize prediction in sklearn.

## 8. Resources

- **Official docs** — https://www.statsmodels.org/stable/ (User Guide is organized by model family:
  regression, GLM, time series, stats).
- **Getting started + formula API** — https://www.statsmodels.org/stable/gettingstarted.html and
  https://www.statsmodels.org/stable/example_formulas.html (the R-style `y ~ x` interface in depth).
- **Examples gallery** — https://www.statsmodels.org/stable/examples/index.html (runnable notebooks
  for OLS, GLM, ARIMA/SARIMAX, diagnostics, and more).
- **Time-series (tsa) guide** — https://www.statsmodels.org/stable/tsa.html (ARIMA/SARIMAX, state
  space, ACF/PACF, stationarity tests).
- **statsmodels vs scikit-learn** — https://www.statsmodels.org/stable/index.html#background
  (the inference-vs-prediction framing that tells you which library a task belongs to).

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def design_columns(formula, levels, intercept=True):
    """The response and the design-matrix column names a formula expands to."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE